# Open Sustainability Analyst as a library

End-to-end **OSA** without Streamlit: load [ClimRetrieve](https://github.com/tobischimanski/ClimRetrieve)
labels, run the same `DocumentAnalyzer` the app uses, score retrieval against expert chunks, then
check how stable scores and citations are.

The older Colab notebook cloned a private fork with a GitHub token. This package is public — install
from GitHub, no token.

## Evaluation

**Retrieval vs ClimRetrieve** on 10 labelled sustainability reports and assessment questions.

Charts:
1. Precision and recall
2. Overlap of ClimRetrieve-relevant chunks and OSA-selected chunks

**Robustness** — repeated runs and two configs (`top_k=5` vs `top_k=10`):
- How stable are OSA scores, and how much do they move with top-k?
- How stable is the retrieved chunk set, and are k=5 chunks contained in k=10?
- How consistently does OSA cite the same chunks, and are citations a subset of retrieval?

All tables are written to `notebooks/output/` as CSV.

Set `RUN_LIVE_ANALYSIS = True` (requires `OPENAI_API_KEY`) for PDFs + embeddings + LLM.
Without a key the notebook still downloads labels and selects the 10 reports.


## 0. Install


In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "report_analyst").exists():
    sys.path.insert(0, str(REPO_ROOT))
elif (REPO_ROOT.parent / "report_analyst").exists():
    REPO_ROOT = REPO_ROOT.parent
    sys.path.insert(0, str(REPO_ROOT))

if IN_COLAB:
    # Public repo — no GitHub token.
    %pip install -q "git+https://github.com/climateandtech/report-analyst.git@notebooks/library-e2e-climretrieve" openpyxl matplotlib requests python-dotenv
else:
    %pip install -q openpyxl matplotlib requests


## 1. Configuration


In [ ]:
import os
import uuid
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

RUN_LIVE_ANALYSIS = bool(os.getenv("OPENAI_API_KEY"))
RUN_ROBUSTNESS = os.getenv("CLIMRETRIEVE_RUN_ROBUSTNESS", "true").lower() == "true"
EVALUATION_ID = uuid.uuid4().hex
N_REPORTS = int(os.getenv("CLIMRETRIEVE_N_REPORTS", "10"))
MAX_QUESTIONS = int(os.getenv("CLIMRETRIEVE_MAX_QUESTIONS", "16"))
N_RUNS = int(os.getenv("CLIMRETRIEVE_N_RUNS", "3"))
ROBUSTNESS_N_REPORTS = int(os.getenv("CLIMRETRIEVE_ROBUSTNESS_N_REPORTS", "1"))
ROBUSTNESS_MAX_QUESTIONS = int(os.getenv("CLIMRETRIEVE_ROBUSTNESS_MAX_QUESTIONS", "5"))
ROBUSTNESS_USE_LLM_SCORING = (
    os.getenv("CLIMRETRIEVE_ROBUSTNESS_LLM_SCORING", "false").lower() == "true"
)
TOP_K_VALUES = [5, 10]
TOP_K_A, TOP_K_B = TOP_K_VALUES
CHUNK_SIZES = [200, 400]
CHUNK_SIZE = CHUNK_SIZES[0]
CHUNK_OVERLAP = 20
K_VALUES = [1, 3, 5, 10]
BINARY_RELEVANCE_MIN = 2
BOOTSTRAP_SAMPLES = 2000
BOOTSTRAP_SEED = 42
QUESTION_SET = "climretrieve"

DATA_DIR = Path("notebooks/data") if Path("notebooks").exists() else Path("data")
default_output_dir = "notebooks/output" if Path("notebooks").exists() else "output"
OUTPUT_DIR = Path(os.getenv("CLIMRETRIEVE_OUTPUT_DIR", default_output_dir))
PDF_DIR = DATA_DIR / "climretrieve_pdfs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_LIVE_ANALYSIS=", RUN_LIVE_ANALYSIS)
print("output=", OUTPUT_DIR.resolve())


## 2. ClimRetrieve expert labels

Public files from [tobischimanski/ClimRetrieve](https://github.com/tobischimanski/ClimRetrieve).
We keep Core-16 questions and reports that also exist as PDFs.


In [ ]:
import pandas as pd
import requests

from report_analyst.core.benchmark.library_eval import (
    filter_core_questions,
    normalize_climretrieve_columns,
    select_labelled_reports,
)

CLIMRETRIEVE_REPO = "tobischimanski/ClimRetrieve"
LABELS_URL = (
    f"https://raw.githubusercontent.com/{CLIMRETRIEVE_REPO}/main/"
    "Expert-Annotated%20Relevant%20Sources%20Dataset/ClimRetrieve_base.xlsx"
)
REPORTS_API = f"https://api.github.com/repos/{CLIMRETRIEVE_REPO}/contents/Reports"

labels_xlsx = DATA_DIR / "ClimRetrieve_base.xlsx"
if not labels_xlsx.exists():
    response = requests.get(LABELS_URL, timeout=60)
    response.raise_for_status()
    labels_xlsx.write_bytes(response.content)

raw_labels = pd.read_excel(labels_xlsx)
labels = filter_core_questions(normalize_climretrieve_columns(raw_labels))
print("Columns:", list(raw_labels.columns))
print("Core-16 rows:", len(labels), "reports:", labels["document"].nunique())
display(labels.head())

listing = requests.get(REPORTS_API, timeout=60)
listing.raise_for_status()
pdf_files = [item["name"] for item in listing.json() if str(item.get("name", "")).lower().endswith(".pdf")]
print("PDFs:", len(pdf_files))

selected_reports = select_labelled_reports(raw_labels, pdf_files, n=N_REPORTS)
display(selected_reports)
if len(selected_reports) < N_REPORTS:
    print(f"Warning: only {len(selected_reports)} labelled reports matched PDFs")
selected_reports.to_csv(OUTPUT_DIR / "selected_reports.csv", index=False)


## 3. Download the labelled PDFs


In [ ]:
from urllib.parse import quote

def download_pdf(filename: str) -> Path:
    target = PDF_DIR / filename
    if target.exists() and target.stat().st_size > 0:
        return target
    url = f"https://raw.githubusercontent.com/{CLIMRETRIEVE_REPO}/main/Reports/{quote(filename)}"
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    target.write_bytes(response.content)
    return target

pdf_paths = {}
for _, row in selected_reports.iterrows():
    path = download_pdf(row["pdf_filename"])
    pdf_paths[row["document"]] = path
    kb = path.stat().st_size // 1024
    print(f"{row['document']}: {path.name} ({kb} KB)")


## 4. End-to-end analysis (library, no frontend)

`DocumentAnalyzer` is the same object Streamlit uses: question set, chunk parameters,
`retrieve_chunks` (retrieval only) or `process_document` (answers, scores, citations).


In [ ]:
from report_analyst.core.analyzer import DocumentAnalyzer
from report_analyst.core.benchmark.library_eval import match_question
from report_analyst.core.question_loader import get_question_loader

DocumentAnalyzer.reset_instance()
analyzer = DocumentAnalyzer()
analyzer.update_question_set(QUESTION_SET)
analyzer.update_parameters(CHUNK_SIZE, CHUNK_OVERLAP, TOP_K_A)

osa_questions = get_question_loader().get_questions(QUESTION_SET)
clim_questions = sorted(labels["question"].dropna().unique())
rows = []
for qid, payload in osa_questions.items():
    clim_q = match_question(payload["text"], clim_questions)
    if not clim_q:
        continue
    number = int(str(qid).rsplit("_", 1)[-1])
    rows.append(
        {
            "osa_question_id": qid,
            "osa_question_number": number,
            "osa_text": payload["text"],
            "climretrieve_question": clim_q,
        }
    )
selected_question_counts = (
    labels[labels["document"].isin(selected_reports["document"])]
    .groupby("question")
    .size()
    .rename("annotated_span_count")
)
question_map_df = (
    pd.DataFrame(rows)
    .drop_duplicates("climretrieve_question")
    .merge(
        selected_question_counts,
        left_on="climretrieve_question",
        right_index=True,
        how="inner",
    )
    .sort_values(["annotated_span_count", "osa_question_number"], ascending=[False, True])
    .head(MAX_QUESTIONS)
    .reset_index(drop=True)
)
display(question_map_df)
question_map_df.to_csv(OUTPUT_DIR / "question_map.csv", index=False)
print("Mapped questions:", len(question_map_df))
print("LLM configured:", analyzer.llm is not None)


In [ ]:
async def collect_analysis(file_path, question_numbers, force_recompute=True, use_llm_scoring=False):
    collected = []
    async for item in analyzer.process_document(
        file_path=str(file_path),
        selected_questions=list(question_numbers),
        use_llm_scoring=use_llm_scoring,
        force_recompute=force_recompute,
    ):
        if "error" in item:
            print(item["error"])
            continue
        if "result" in item:
            collected.append(item)
    return collected


if RUN_LIVE_ANALYSIS and question_map_df.empty:
    raise RuntimeError("No OSA questions mapped onto ClimRetrieve Core-16 questions.")

if RUN_LIVE_ANALYSIS:
    demo_doc = selected_reports.iloc[0]["document"]
    demo_pdf = pdf_paths[demo_doc]
    demo_q = question_map_df.iloc[0]
    print("Demo:", demo_doc, demo_q["osa_question_id"])
    retrieved = await analyzer.retrieve_chunks(str(demo_pdf), demo_q["osa_text"], top_k=TOP_K_A)
    print(f"Retrieved {len(retrieved)} chunks")
    for i, chunk in enumerate(retrieved[:3], start=1):
        score = chunk.get("score", chunk.get("similarity_score", 0.0))
        print(f"\n[{i}] score={score:.3f}\n{chunk['text'][:400]}")
    analysis = await collect_analysis(demo_pdf, [int(demo_q["osa_question_number"])])
    if analysis:
        result = analysis[0]["result"]
        print("SCORE:", result.get("SCORE"))
        print("SOURCES:", result.get("SOURCES"))
        print("ANSWER:", str(result.get("ANSWER", ""))[:500])
else:
    print("Skipping live analysis (set OPENAI_API_KEY and RUN_LIVE_ANALYSIS=True).")


## 5. Paper retrieval metrics vs ClimRetrieve

We compare chunk sizes **200 and 400 side by side**. Candidate chunks come from embedding retrieval, but a benchmark hit requires the complete ordered ClimRetrieve span:

1. exact normalized token sequence; or
2. the retrieved chunk contains the complete ground-truth sequence; or
3. two adjacent retrieved chunks reconstruct the complete sequence after their duplicated overlap is removed.

Partial token overlap, METEOR, BERTScore, and embedding similarity are diagnostic signals only; they do not create a hit. If a ground-truth evidence span contains only the retrieved chunk, the relation is recorded but evidence is incomplete.

Matching is restricted to evidence spans for the same report and question. One retrieved chunk may contain several such spans, so the exported match table preserves one row per chunk–span relation. Ranking still treats the chunk as one item and assigns it the maximum matched relevance grade; evidence recall counts unique matched spans. Only report–question pairs with expert evidence judgments enter the ranking evaluation.

### How to read the metrics

- **nDCG@k** is the primary ranking metric. It rewards highly relevant evidence near the top and uses ClimRetrieve's grades 0–3:

  \[
  \operatorname{nDCG}@k =
  \frac{1}{\operatorname{IDCG}@k}
  \sum_{i=1}^{k}\frac{\mathrm{rel}_i}{\log_2(i+1)} .
  \]

- **Recall@k** measures how much of the relevant evidence was found; **Precision@k** measures how much of the retrieved list was relevant.
- **MAP** summarizes the ranks of all relevant items, while **MRR** considers only the first relevant item.
- **Complete-set hit rate** requires all relevant evidence for a query to be retrieved.
- **Strict evidence recall** includes exact, containing, and reconstructed split matches; partial matches do not count.
- **Retrieved-context coverage** indicates focus: lower values mean that more unrelated text surrounds the matched evidence.
- **Paired deltas** are calculated per report-question pair as the 400-token result minus the 200-token result.
- **95% confidence intervals** use 2,000 query-level bootstrap samples with seed 42.

Binary metrics treat ClimRetrieve grades **2 and 3 as relevant**. Grade 1 remains part of graded nDCG.

References: [ClimRetrieve (EMNLP 2024)](https://aclanthology.org/2024.emnlp-main.969/), [BEIR](https://arxiv.org/abs/2104.08663), [BERTScore](https://arxiv.org/abs/1904.09675), and [METEOR](https://www.cs.cmu.edu/~alavie/papers/BanerjeeLavie2005-final.pdf).


In [ ]:
from report_analyst.core.benchmark.library_eval import (
    bootstrap_macro_intervals,
    build_chunk_dataset_rows,
    build_climretrieve_answer_rows,
    build_ground_truth_rows,
    build_osa_retrieval_rows,
    build_overlap_table,
    build_retrieval_match_table,
    query_match_metrics,
    ranked_query_metrics,
    summarize_query_match_metrics,
    summarize_ranked_query_metrics,
)

gt = build_ground_truth_rows(raw_labels, selected_reports["document"].tolist())
gt = gt[gt["question"].isin(set(question_map_df["climretrieve_question"]))]
gt.to_csv(OUTPUT_DIR / "climretrieve_ground_truth.csv", index=False)
expert_answers = build_climretrieve_answer_rows(
    raw_labels,
    selected_reports["document"].tolist(),
    question_map_df["climretrieve_question"].tolist(),
)
expert_answer_lookup = {
    (row.document, row.question): row for row in expert_answers.itertuples(index=False)
}
expert_answers.to_csv(OUTPUT_DIR / "climretrieve_answers.csv", index=False)
print("Ground-truth rows:", len(gt), "queries:", gt["query_id"].nunique())
print("Explicit expert yes/no answers:", expert_answers["expert_yes_no"].notna().sum())
evaluation_pairs = (
    gt[["document", "question"]]
    .drop_duplicates()
    .merge(
        question_map_df[["climretrieve_question", "osa_text"]],
        left_on="question",
        right_on="climretrieve_question",
        how="inner",
    )
)
print("Judged report-question pairs evaluated:", len(evaluation_pairs))

retrieval_frames = []
retrieval_match_frames = []
corpus_match_frames = []
overlap_frames = []
chunk_dataset_frames = []
if RUN_LIVE_ANALYSIS:
    for chunk_size in CHUNK_SIZES:
        config_id = f"cs{chunk_size}"
        analyzer.update_parameters(chunk_size, CHUNK_OVERLAP, max(K_VALUES))
        retrieved_by_query = {}
        corpus_by_query = {}
        all_chunks_by_document = {}
        for _, pair in evaluation_pairs.iterrows():
            document = pair["document"]
            pdf = pdf_paths[document]
            chunks = await analyzer.retrieve_chunks(
                str(pdf), pair["osa_text"], top_k=max(K_VALUES)
            )
            retrieved_by_query[(document, pair["question"])] = chunks
            if document not in all_chunks_by_document:
                all_chunks_by_document[document] = analyzer.cache_manager.get_document_chunks(
                    str(pdf), chunk_size, CHUNK_OVERLAP
                )
                chunk_dataset_frames.append(
                    build_chunk_dataset_rows(
                        all_chunks_by_document[document],
                        document=document,
                        pdf_filename=pdf.name,
                        config_id=config_id,
                        chunk_size=chunk_size,
                        chunk_overlap=CHUNK_OVERLAP,
                    )
                )
            corpus_by_query[(document, pair["question"])] = all_chunks_by_document[document]
        config_rows = build_osa_retrieval_rows(retrieved_by_query, gt)
        corpus_rows = build_osa_retrieval_rows(corpus_by_query, gt)
        if config_rows.empty:
            continue
        for frame in (config_rows, corpus_rows):
            frame["config_id"] = config_id
            frame["chunk_size"] = chunk_size
            frame["chunk_overlap"] = CHUNK_OVERLAP
            frame["top_k"] = max(K_VALUES)
        config_matches = build_retrieval_match_table(config_rows, gt)
        corpus_matches = build_retrieval_match_table(corpus_rows, gt)
        retrieval_frames.append(config_rows)
        retrieval_match_frames.append(config_matches)
        corpus_match_frames.append(corpus_matches)
        config_overlap = build_overlap_table(gt, config_rows, matches=config_matches)
        config_overlap["config_id"] = config_id
        config_overlap["chunk_size"] = chunk_size
        overlap_frames.append(config_overlap)

osa_retrieval = pd.concat(retrieval_frames, ignore_index=True) if retrieval_frames else pd.DataFrame()
retrieval_matches = (
    pd.concat(retrieval_match_frames, ignore_index=True) if retrieval_match_frames else pd.DataFrame()
)
corpus_matches = pd.concat(corpus_match_frames, ignore_index=True) if corpus_match_frames else pd.DataFrame()
overlap = pd.concat(overlap_frames, ignore_index=True) if overlap_frames else pd.DataFrame()
generated_chunk_datasets = (
    pd.concat(chunk_dataset_frames, ignore_index=True)
    if chunk_dataset_frames
    else pd.DataFrame()
)
if not generated_chunk_datasets.empty:
    display(
        generated_chunk_datasets.groupby(
            ["document", "config_id", "chunk_size", "chunk_overlap"]
        )
        .size()
        .rename("n_chunks")
        .reset_index()
    )
paper_query_ranking = pd.DataFrame()
paper_ranking_summary = pd.DataFrame()
paper_query_matches = pd.DataFrame()
paper_match_summary = pd.DataFrame()
paper_paired_deltas = pd.DataFrame()
paper_intervals = pd.DataFrame()
metrics_df = pd.DataFrame()
chunk_comparison = pd.DataFrame()
if not osa_retrieval.empty:
    osa_retrieval.to_csv(OUTPUT_DIR / "osa_retrieval.csv", index=False)
    overlap.to_csv(OUTPUT_DIR / "chunk_overlap.csv", index=False)
    retrieval_matches.to_csv(OUTPUT_DIR / "retrieval_evidence_matches.csv", index=False)
    corpus_matches.to_csv(OUTPUT_DIR / "corpus_evidence_matches.csv", index=False)
    paper_query_ranking = ranked_query_metrics(
        osa_retrieval,
        gt,
        matches=retrieval_matches,
        corpus_matches=corpus_matches,
        k_values=K_VALUES,
        binary_relevance_min=BINARY_RELEVANCE_MIN,
    )
    paper_ranking_summary = summarize_ranked_query_metrics(paper_query_ranking)
    paper_query_matches = query_match_metrics(
        osa_retrieval, gt, matches=retrieval_matches
    )
    paper_match_summary = summarize_query_match_metrics(paper_query_matches)
    delta_frames = []
    for metric in ["ndcg", "recall", "precision", "f1", "hit", "complete_set_hit"]:
        wide = paper_query_ranking.pivot(
            index=["query_id", "document", "question", "k"],
            columns="chunk_size",
            values=metric,
        ).reset_index()
        if set(CHUNK_SIZES).issubset(wide.columns):
            wide["base_metric"] = metric
            wide["chunk_size_a"] = CHUNK_SIZES[0]
            wide["chunk_size_b"] = CHUNK_SIZES[1]
            wide["delta_b_minus_a"] = (
                wide[CHUNK_SIZES[1]].astype(float)
                - wide[CHUNK_SIZES[0]].astype(float)
            )
            delta_frames.append(wide)
    paper_paired_deltas = pd.concat(delta_frames, ignore_index=True) if delta_frames else pd.DataFrame()
    ranking_intervals = bootstrap_macro_intervals(
        paper_query_ranking,
        ["precision", "recall", "f1", "ndcg", "hit", "complete_set_hit"],
        group_columns=["config_id", "chunk_size", "k"],
        n_bootstrap=BOOTSTRAP_SAMPLES,
        seed=BOOTSTRAP_SEED,
    )
    match_intervals = bootstrap_macro_intervals(
        paper_query_matches,
        ["strict_precision", "strict_recall", "complete_set_hit"],
        group_columns=["config_id", "chunk_size"],
        n_bootstrap=BOOTSTRAP_SAMPLES,
        seed=BOOTSTRAP_SEED,
    )
    paired_intervals = (
        bootstrap_macro_intervals(
            paper_paired_deltas,
            ["delta_b_minus_a"],
            group_columns=["base_metric", "k"],
            n_bootstrap=BOOTSTRAP_SAMPLES,
            seed=BOOTSTRAP_SEED,
        )
        if not paper_paired_deltas.empty
        else pd.DataFrame()
    )
    paper_intervals = pd.concat(
        [ranking_intervals, match_intervals, paired_intervals], ignore_index=True
    )
    metrics_df = paper_ranking_summary.melt(
        id_vars=["config_id", "chunk_size", "chunk_overlap", "top_k", "k", "n_queries"],
        value_vars=["precision", "recall", "f1", "ndcg", "hit_rate", "complete_set_hit_rate", "MAP", "MRR"],
        var_name="metric",
        value_name="value",
    )
    example_query = osa_retrieval["query_id"].iloc[0]
    chunk_comparison = osa_retrieval[
        (osa_retrieval["query_id"] == example_query) & (osa_retrieval["position"] <= 5)
    ].copy()
    comparison_matches = (
        retrieval_matches[retrieval_matches["query_id"] == example_query]
        .groupby(["config_id", "chunk_size", "query_id", "retrieval_position"])
        .agg(
            matched_evidence_count=("ground_truth_chunk_id", "nunique"),
            chunk_relevance=("relevance_grade", "max"),
            matched_evidence_ids=(
                "ground_truth_chunk_id",
                lambda values: " | ".join(sorted({str(value) for value in values})),
            ),
            match_relations=(
                "match_relation",
                lambda values: " | ".join(sorted({str(value) for value in values})),
            ),
        )
        .reset_index()
    )
    chunk_comparison = chunk_comparison.merge(
        comparison_matches,
        left_on=["config_id", "chunk_size", "query_id", "position"],
        right_on=["config_id", "chunk_size", "query_id", "retrieval_position"],
        how="left",
    )
    chunk_comparison["matched_evidence_count"] = (
        chunk_comparison["matched_evidence_count"].fillna(0).astype(int)
    )
    chunk_comparison["chunk_relevance"] = chunk_comparison["chunk_relevance"].fillna(0)
    chunk_comparison["chunk_excerpt"] = chunk_comparison["chunk_text"].str.slice(0, 240)

    display(
        paper_ranking_summary.pivot(
            index="k",
            columns="chunk_size",
            values=["ndcg", "recall", "precision", "MAP", "MRR"],
        ).round(4)
    )
    display(paper_match_summary.round(4))
    if not paper_paired_deltas.empty:
        display(
            paper_paired_deltas.groupby(["base_metric", "k"])["delta_b_minus_a"]
            .agg(["count", "mean", "std", "median", "min", "max"])
            .round(4)
        )
    display(
        chunk_comparison.pivot(
            index="position",
            columns="chunk_size",
            values=[
                "score",
                "chunk_relevance",
                "matched_evidence_count",
                "match_relations",
                "matched_evidence_ids",
                "chunk_excerpt",
            ],
        )
    )
else:
    print("No retrieval rows. Set RUN_LIVE_ANALYSIS=True to compute paper metrics.")


### Side-by-side ranking and chunk-matching charts

The first figure shows ranking quality as the retrieval cutoff increases.

The **evidence coverage and context focus** chart then compares three rates with different denominators:

- strict evidence recall: the fraction of unique ClimRetrieve evidence spans found;
- complete-set hit rate: the fraction of questions for which every evidence span was found;
- retrieved-context coverage: the fraction of each accepted retrieved chunk occupied by the matched evidence (higher means less surrounding text).

The final **accepted match relations** chart is a count, not a quality score. Each row in the match table is one accepted chunk–evidence-span relationship. A chunk containing two query-specific evidence spans therefore contributes two relationships. Unmatched and partial-overlap chunks are omitted.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if paper_ranking_summary.empty or paper_match_summary.empty:
    print("Charts skipped until retrieval rows exist.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
    for metric, ax in zip(["ndcg", "recall", "precision"], axes, strict=True):
        for chunk_size, group in paper_ranking_summary.groupby("chunk_size"):
            group = group.sort_values("k")
            ax.plot(group["k"], group[metric], marker="o", label=f"{chunk_size} tokens")
            ci = paper_intervals[
                (paper_intervals["chunk_size"] == chunk_size)
                & (paper_intervals["metric"] == metric)
                & paper_intervals["k"].notna()
            ].sort_values("k")
            if len(ci) == len(group):
                ax.fill_between(ci["k"], ci["ci_low"], ci["ci_high"], alpha=0.15)
        ax.set_xlabel("Retrieval cutoff k")
        ax.set_ylabel(metric)
        ax.set_title(f"Macro {metric}@k")
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.legend()
    fig.suptitle("Ranking quality by chunk size (95% query-bootstrap CI)")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "paper_ranking_by_chunk_size.png", dpi=150)
    plt.show()

    quality_columns = [
        ("macro_strict_recall", "Strict evidence recall"),
        ("complete_set_hit_rate", "Complete-set hit rate"),
        ("mean_hit_retrieved_coverage", "Retrieved-context coverage"),
    ]
    x = np.arange(len(quality_columns))
    width = 0.8 / len(CHUNK_SIZES)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for offset, chunk_size in enumerate(CHUNK_SIZES):
        row = paper_match_summary[paper_match_summary["chunk_size"] == chunk_size].iloc[0]
        values = [row[column] for column, _ in quality_columns]
        ax.bar(x + offset * width, values, width, label=f"{chunk_size} tokens")
    ax.set_xticks(x + width * (len(CHUNK_SIZES) - 1) / 2)
    ax.set_xticklabels([label for _, label in quality_columns])
    ax.set_ylabel("Macro mean")
    ax.set_ylim(0, 1.05)
    ax.set_title("Evidence coverage and context focus by chunk size")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "paper_match_quality_by_chunk_size.png", dpi=150)
    plt.show()

    if retrieval_matches.empty:
        print("Accepted match-relation chart skipped: no evidence spans matched.")
    else:
        relation_counts = (
            retrieval_matches.groupby(["chunk_size", "match_relation"])
            .size()
            .unstack(fill_value=0)
        )
        relation_counts.plot(kind="bar", stacked=True, figsize=(10, 4.5))
        plt.xlabel("Chunk size (tokens)")
        plt.ylabel("Accepted chunk–evidence relationships")
        plt.title("Accepted evidence-match relations by chunk size")
        plt.legend(title="Relation")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "paper_match_relations.png", dpi=150)
        plt.show()


## 6. Bounded robustness run and configuration heatmaps

To keep interactive execution manageable, robustness defaults to one report, five judged questions, and three runs for every chunk-size/top-k combination. Each report/configuration/run is processed in one document pass instead of restarting the document for every question. Retrieval uses cached chunks and embeddings; optional LLM chunk rescoring is off by default because it doubles the model work and is not needed to test embedding selection stability.

Override `CLIMRETRIEVE_ROBUSTNESS_N_REPORTS`, `CLIMRETRIEVE_ROBUSTNESS_MAX_QUESTIONS`, `CLIMRETRIEVE_N_RUNS`, or set `CLIMRETRIEVE_ROBUSTNESS_LLM_SCORING=true` for a larger experiment. Per-report heatmaps place all configurations in columns and questions in rows.


In [ ]:
from report_analyst.core.benchmark.library_eval import (
    build_analysis_run_rows,
    citation_consistency,
    citation_subset_rate,
    combine_analysis_run_rows,
    pairwise_chunk_selection,
    question_configuration_summary,
    retrieved_chunk_consistency,
    score_distribution_summary,
    score_stability,
    topk_retrieved_containment,
    topk_score_delta,
    yes_no_answer_comparison,
)

robustness_rows = []
chunk_score_rows = []
all_result_rows = []
if RUN_LIVE_ANALYSIS and RUN_ROBUSTNESS:
    analyzer.use_cache = True
    configs = [
        (f"cs{chunk_size}_k{top_k}", chunk_size, top_k)
        for chunk_size in CHUNK_SIZES
        for top_k in TOP_K_VALUES
    ]
    robustness_reports = selected_reports.head(ROBUSTNESS_N_REPORTS)
    for _, report in robustness_reports.iterrows():
        pdf = pdf_paths[report["document"]]
        gt_doc = gt[gt["document"] == report["document"]]
        report_questions = question_map_df[
            question_map_df["climretrieve_question"].isin(gt_doc["question"])
        ].head(ROBUSTNESS_MAX_QUESTIONS)
        question_numbers = report_questions["osa_question_number"].astype(int).tolist()
        if not question_numbers:
            continue
        for config_id, chunk_size, top_k in configs:
            analyzer.update_parameters(chunk_size, CHUNK_OVERLAP, top_k)
            for run_id in range(1, N_RUNS + 1):
                print(
                    f"Robustness: {report['document']} {config_id} "
                    f"run {run_id}/{N_RUNS} ({len(question_numbers)} questions)"
                )
                items = await collect_analysis(
                    pdf,
                    question_numbers,
                    force_recompute=True,
                    use_llm_scoring=ROBUSTNESS_USE_LLM_SCORING,
                )
                results_by_question = {
                    int(item["question_number"]): item["result"] for item in items
                }
                for _, qrow in report_questions.iterrows():
                    question_number = int(qrow["osa_question_number"])
                    result = results_by_question.get(question_number, {})
                    gt_q = gt_doc[gt_doc["question"] == qrow["climretrieve_question"]]
                    expert = expert_answer_lookup.get(
                        (report["document"], qrow["climretrieve_question"])
                    )
                    context = {
                        "evaluation_id": EVALUATION_ID,
                        "document": report["document"],
                        "pdf_filename": report["pdf_filename"],
                        "question": qrow["climretrieve_question"],
                        "osa_question_id": qrow["osa_question_id"],
                        "config_id": config_id,
                        "top_k": top_k,
                        "chunk_size": chunk_size,
                        "chunk_overlap": CHUNK_OVERLAP,
                        "model": analyzer.default_model,
                        "run_id": run_id,
                        "expert_answer": getattr(expert, "expert_answer", None),
                        "expert_yes_no": getattr(expert, "expert_yes_no", None),
                    }
                    answer_row, chunk_rows = build_analysis_run_rows(result, gt_q, context)
                    robustness_rows.append(answer_row)
                    chunk_score_rows.extend(chunk_rows)
                    all_result_rows.extend(combine_analysis_run_rows(answer_row, chunk_rows))

robustness = pd.DataFrame(robustness_rows)
chunk_scores = pd.DataFrame(chunk_score_rows)
all_results = pd.DataFrame(all_result_rows)
stability = pd.DataFrame()
citations = pd.DataFrame()
retrieved_sets = pd.DataFrame()
selection_pairs = pd.DataFrame()
containment = pd.DataFrame()
deltas = pd.DataFrame()
answer_ranges = pd.DataFrame()
chunk_score_ranges = pd.DataFrame()
selection_ranges = pd.DataFrame()
yes_no_detail = pd.DataFrame()
yes_no_metrics = pd.DataFrame()
question_config_summary = pd.DataFrame()
if robustness.empty:
    print("Robustness skipped.")
else:
    stability = score_stability(robustness)
    citations = citation_consistency(robustness)
    retrieved_sets = retrieved_chunk_consistency(robustness)
    selection_pairs = pairwise_chunk_selection(robustness)
    low_config = f"cs{CHUNK_SIZES[0]}_k{TOP_K_VALUES[0]}"
    high_config = f"cs{CHUNK_SIZES[0]}_k{TOP_K_VALUES[1]}"
    containment = topk_retrieved_containment(robustness, low_config, high_config)
    deltas = topk_score_delta(stability, low_config, high_config)
    distribution_groups = ("chunk_size", "top_k", "config_id")
    answer_ranges = score_distribution_summary(robustness, "answer_score", distribution_groups)
    chunk_score_ranges = score_distribution_summary(
        chunk_scores, "llm_score", (*distribution_groups, "is_evidence")
    )
    selection_ranges = score_distribution_summary(selection_pairs, "selection_jaccard", distribution_groups)
    yes_no_detail, yes_no_metrics = yes_no_answer_comparison(robustness)
    question_config_summary = question_configuration_summary(robustness)
    print(f"Citations contained in retrieval: {citation_subset_rate(robustness):.1%}")
    display(question_config_summary)
    display(answer_ranges)
    display(chunk_score_ranges)
    display(selection_ranges)
    display(yes_no_metrics)


In [ ]:
import textwrap


def draw_config_heatmap(ax, matrix, title, config_labels, dividers):
    image = ax.imshow(matrix, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Configuration (chunk size / top-k)")
    ax.set_xticks(range(len(config_labels)), config_labels, rotation=35, ha="right")
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    for row_index in range(len(matrix.index)):
        for column_index in range(len(matrix.columns)):
            value = matrix.iloc[row_index, column_index]
            label = "–" if pd.isna(value) else f"{value:.2f}"
            ax.text(column_index, row_index, label, ha="center", va="center", fontsize=8)
    for divider in dividers:
        ax.axvline(divider, color="white", linewidth=2)
    return image


if robustness.empty:
    print("Robustness charts skipped.")
else:
    config_metadata = (
        robustness[["config_id", "chunk_size", "top_k"]]
        .drop_duplicates()
        .sort_values(["chunk_size", "top_k", "config_id"])
    )
    config_order = config_metadata["config_id"].tolist()
    config_labels = [
        f"{row.chunk_size} / k={row.top_k}"
        for row in config_metadata.itertuples(index=False)
    ]
    chunk_sizes = config_metadata["chunk_size"].tolist()
    dividers = [
        index - 0.5
        for index in range(1, len(chunk_sizes))
        if chunk_sizes[index] != chunk_sizes[index - 1]
    ]
    selection_summary = retrieved_sets.merge(
        config_metadata,
        on="config_id",
        how="left",
    )
    for document in sorted(robustness["document"].dropna().unique()):
        answer_document = question_config_summary[
            question_config_summary["document"] == document
        ].copy()
        selection_document = selection_summary[
            selection_summary["document"] == document
        ].copy()
        question_order = answer_document["question"].drop_duplicates().tolist()
        question_labels = {
            question: textwrap.shorten(question, width=62, placeholder="…")
            for question in question_order
        }

        def heatmap_matrix(frame, value_column):
            matrix = frame.pivot(
                index="question",
                columns="config_id",
                values=value_column,
            ).reindex(index=question_order, columns=config_order)
            return matrix.rename(index=question_labels)

        matrices = [
            (heatmap_matrix(answer_document, "answer_accuracy"), "Answer correctness rate"),
            (heatmap_matrix(answer_document, "answer_agreement"), "Answer agreement across runs"),
            (
                heatmap_matrix(selection_document, "retrieved_jaccard"),
                "Retrieved-chunk stability across runs",
            ),
        ]
        height = max(4.5, 0.55 * len(question_order))
        fig, axes = plt.subplots(1, 3, figsize=(18, height), sharey=True)
        for ax, (matrix, title) in zip(axes, matrices, strict=True):
            image = draw_config_heatmap(
                ax,
                matrix,
                title,
                config_labels,
                dividers,
            )
            fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Rate (0–1)")
        axes[0].set_ylabel("Question")
        fig.suptitle(f"Answer and chunk-selection comparison — {document}")
        fig.tight_layout()
        output_name = Path(document).stem.replace(" ", "_")
        fig.savefig(OUTPUT_DIR / f"configuration_heatmaps_{output_name}.png", dpi=150)
        plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    plot_specs = [
        (robustness, "answer_score", "Answer score", axes[0]),
        (chunk_scores, "llm_score", "Selected-chunk LLM score", axes[1]),
        (selection_pairs, "selection_jaccard", "Selected-chunk consistency", axes[2]),
    ]
    for frame, value, title, ax in plot_specs:
        groups = [(name, group[value].dropna()) for name, group in frame.groupby("config_id")]
        groups = [(name, values) for name, values in groups if not values.empty]
        if groups:
            ax.boxplot([values for _, values in groups], tick_labels=[name for name, _ in groups])
        ax.set_title(title)
        ax.set_xlabel("Configuration")
        ax.set_ylabel(value)
        ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "robustness_boxplots.png", dpi=150)
    plt.show()


## 7. CSV exports


In [ ]:
from report_analyst.core.benchmark.library_eval import write_eval_csvs

frames = {
    "selected_reports": selected_reports,
    "question_map": question_map_df,
    "climretrieve_ground_truth": gt,
    "climretrieve_answers": expert_answers,
    "climretrieve_labels_core16": labels,
}
if not osa_retrieval.empty:
    frames["osa_retrieval"] = osa_retrieval
    frames["generated_chunk_datasets"] = generated_chunk_datasets
    frames["retrieval_evidence_matches"] = retrieval_matches
    frames["corpus_evidence_matches"] = corpus_matches
    frames["chunk_overlap"] = overlap
    frames["retrieval_metrics"] = metrics_df
    frames["paper_query_ranking_metrics"] = paper_query_ranking
    frames["paper_ranking_summary"] = paper_ranking_summary
    frames["paper_query_match_metrics"] = paper_query_matches
    frames["paper_match_summary"] = paper_match_summary
    frames["paper_paired_chunk_size_deltas"] = paper_paired_deltas
    frames["paper_metric_intervals"] = paper_intervals
    frames["paper_chunk_comparison"] = chunk_comparison
if not robustness.empty:
    frames["all_results"] = all_results
    frames["analysis_runs"] = robustness
    frames["chunk_scores"] = chunk_scores
    frames["answer_score_stability"] = stability
    frames["answer_score_ranges"] = answer_ranges
    frames["chunk_llm_score_ranges"] = chunk_score_ranges
    frames["chunk_selection_pairs"] = selection_pairs
    frames["chunk_selection_ranges"] = selection_ranges
    frames["citation_consistency"] = citations
    frames["retrieved_chunk_consistency"] = retrieved_sets
    frames["topk_retrieved_containment"] = containment
    frames["topk_answer_score_delta"] = deltas
    frames["yes_no_answer_comparison"] = yes_no_detail
    frames["yes_no_answer_metrics"] = yes_no_metrics
    frames["question_configuration_summary"] = question_config_summary

written = write_eval_csvs(frames, OUTPUT_DIR)
for name, path in written.items():
    print(f"{name}: {path} ({path.stat().st_size} bytes)")
